In [1]:
import sys
sys.path.insert(0, "../src")

import json
import pandas as pd
import networkx as nx
from pathlib import Path
from graph_visualizer import (
    visualize_static, 
    visualize_interactive, 
    query_graph_by_skill, 
    get_cross_domain_skills
)

# Load master graph from JSON-LD
master_path = "../outputs/skill_graphs/DRISHTI_master_graph.json"

with open(master_path, 'r') as f:
    data = json.load(f)

G = nx.DiGraph()

for node in data["nodes"]:
    node_id = node.pop("id")
    G.add_node(node_id, **node)

for edge in data["edges"]:
    src = edge.pop("source")
    tgt = edge.pop("target")
    G.add_edge(src, tgt, **edge)

print(f"✅ Master Graph Loaded")
print(f"   Nodes: {G.number_of_nodes()}")
print(f"   Edges: {G.number_of_edges()}")

✅ Master Graph Loaded
   Nodes: 38
   Edges: 31


In [2]:
output_dir = Path("../outputs/visualizations")
output_dir.mkdir(parents=True, exist_ok=True)

# Color by domain
visualize_static(G, output_dir / "skill_graph_by_domain.png", 
                 color_by="domain", 
                 title="DRISHTI Skill Graph — Color by Domain")

# Color by robot skill
visualize_static(G, output_dir / "skill_graph_by_skill.png", 
                 color_by="skill", 
                 title="DRISHTI Skill Graph — Color by Robot Skill")

✅ Saved static graph: ../outputs/visualizations/skill_graph_by_domain.png


/Users/suryanshsingh/Desktop/DRISHTI/notebooks/../src/graph_visualizer.py:84: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


✅ Saved static graph: ../outputs/visualizations/skill_graph_by_skill.png


In [3]:
visualize_interactive(G, output_dir / "skill_graph_interactive.html", color_by="skill")

print("\n🎯 Open this file in browser:")
print(f"   {(output_dir / 'skill_graph_interactive.html').resolve()}")

✅ Saved interactive graph: ../outputs/visualizations/skill_graph_interactive.html

🎯 Open this file in browser:
   /Users/suryanshsingh/Desktop/DRISHTI/outputs/visualizations/skill_graph_interactive.html


In [4]:
target_skill = "dexterous_fine_manipulation"

results = query_graph_by_skill(G, target_skill)

print(f"🔎 Found {len(results)} segments matching skill: '{target_skill}'\n")
df_query = pd.DataFrame(results)
display(df_query)

🔎 Found 19 segments matching skill: 'dexterous_fine_manipulation'



,node_id,domain,video,frames,intent,kinematic,confidence
0,artificial_jewellery::video_20260404_115542::s...,artificial_jewellery,video_20260404_115542,150-690,jewellery_assembling,pinch,0.8
1,artificial_jewellery::video_20260404_115542::s...,artificial_jewellery,video_20260404_115542,3060-11520,jewellery_assembling,pinch,0.8
2,artificial_jewellery::video_20260404_120254::s...,artificial_jewellery,video_20260404_120254,0-9330,jewellery_assembling,pinch,0.8
3,artificial_jewellery::video_20260404_121005::s...,artificial_jewellery,video_20260404_121005,0-1740,jewellery_assembling,pinch,0.8
4,artificial_jewellery::video_20260404_121005::s...,artificial_jewellery,video_20260404_121005,2370-2580,jewellery_assembling,pinch,0.9
5,artificial_jewellery::video_20260404_121903::s...,artificial_jewellery,video_20260404_121903,0-120,jewellery_assembling,pinch,0.8
6,artificial_jewellery::video_20260404_121903::s...,artificial_jewellery,video_20260404_121903,510-810,jewellery_assembling,pinch,0.9
7,artificial_jewellery::video_20260404_121903::s...,artificial_jewellery,video_20260404_121903,1080-1140,jewellery_assembling,pinch,0.9
8,artificial_jewellery::video_20260404_121903::s...,artificial_jewellery,video_20260404_121903,1440-1680,jewellery_assembling,pinch,0.9
9,artificial_jewellery::video_20260404_121903::s...,artificial_jewellery,video_20260404_121903,1890-2910,jewellery_assembling,pinch,0.8


In [5]:
cross_skills = get_cross_domain_skills(G)

print("🔗 Skills That Transfer Across Domains:\n")
for skill, domains in cross_skills.items():
    print(f"  ✨ {skill}")
    print(f"     → Found in: {', '.join(domains)}")
    print()

print(f"\n💡 Total transferable skills: {len(cross_skills)}")
print(f"💡 This proves: a robot trained on one domain can leverage skills in another!")

🔗 Skills That Transfer Across Domains:


💡 Total transferable skills: 0
💡 This proves: a robot trained on one domain can leverage skills in another!


In [6]:
# Aggregate stats from all enriched CSVs
all_enriched = list(Path("../outputs/robot_skills").rglob("*_enriched.csv"))

dfs = [pd.read_csv(p) for p in all_enriched if p.stat().st_size > 0]
combined = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print("=" * 60)
print("📊 DRISHTI PIPELINE STATISTICS")
print("=" * 60)
print(f"\n📹 Videos processed: {len(all_enriched)}")
print(f"🎬 Total segments extracted: {len(combined)}")
print(f"🤖 Unique robot skills identified: {combined['robot_skill'].nunique()}")
print(f"🎯 Unique intents identified: {combined['intent'].nunique()}")

print("\n--- DOMAIN BREAKDOWN ---")
print(combined.groupby("domain").size())

print("\n--- TOP 5 ROBOT SKILLS ---")
print(combined["robot_skill"].value_counts().head())

print("\n--- TOP 5 INTENTS ---")
print(combined["intent"].value_counts().head())

print("\n--- SKILL COMPLEXITY DISTRIBUTION ---")
print(combined["skill_complexity"].value_counts())

print("\n--- BIMANUAL vs SINGLE-HAND ---")
print(combined["bimanual"].value_counts())

📊 DRISHTI PIPELINE STATISTICS

📹 Videos processed: 7
🎬 Total segments extracted: 31
🤖 Unique robot skills identified: 4
🎯 Unique intents identified: 3

--- DOMAIN BREAKDOWN ---
domain
artificial_jewellery    30
shop                     1
dtype: int64

--- TOP 5 ROBOT SKILLS ---
robot_skill
dexterous_fine_manipulation    19
precision_sorting               7
precision_placement             4
handover_interaction            1
Name: count, dtype: int64

--- TOP 5 INTENTS ---
intent
jewellery_assembling     23
jewellery_sorting         7
shop_customer_service     1
Name: count, dtype: int64

--- SKILL COMPLEXITY DISTRIBUTION ---
skill_complexity
high      19
medium    12
Name: count, dtype: int64

--- BIMANUAL vs SINGLE-HAND ---
bimanual
False    19
True     12
Name: count, dtype: int64


In [7]:
# Show one segment with all 3 layers — the DRISHTI showcase
sample = combined.iloc[0].to_dict()

print("=" * 60)
print("🎓 DRISHTI SAMPLE OUTPUT — Complete Skill Node")
print("=" * 60)

print(f"\n📹 Video: {sample.get('video_stem')}")
print(f"🏷️  Domain: {sample.get('domain')}")
print(f"⏱️  Frames: {sample.get('start_frame')}-{sample.get('end_frame')}")

print("\n┌─────────────────────────────────────────────────┐")
print(f"│ 🤲 KINEMATIC : {sample.get('grasp_used'):<33}│")
print(f"│ 🎯 INTENT    : {sample.get('intent'):<33}│")
print(f"│ 🤖 ROBOT SKILL: {sample.get('robot_skill'):<32}│")
print("└─────────────────────────────────────────────────┘")

print(f"\n📝 Description: {sample.get('intent_description')}")
print(f"⚙️  Complexity: {sample.get('skill_complexity')}")
print(f"🎮 DOF Required: {sample.get('dof_required')}")
print(f"👐 Bimanual: {sample.get('bimanual')}")

🎓 DRISHTI SAMPLE OUTPUT — Complete Skill Node

📹 Video: video_20260405_163219_edit
🏷️  Domain: shop
⏱️  Frames: 720-5910

┌─────────────────────────────────────────────────┐
│ 🤲 KINEMATIC : partial_grip                     │
│ 🎯 INTENT    : shop_customer_service            │
│ 🤖 ROBOT SKILL: handover_interaction            │
└─────────────────────────────────────────────────┘

📝 Description: Interacting with customers, handing over items
⚙️  Complexity: medium
🎮 DOF Required: 5
👐 Bimanual: True


In [8]:
final_summary = {
    "project": "DRISHTI",
    "total_videos": len(all_enriched),
    "total_segments": len(combined),
    "unique_robot_skills": combined['robot_skill'].nunique(),
    "unique_intents": combined['intent'].nunique(),
    "domains": combined['domain'].unique().tolist(),
    "cross_domain_transferable_skills": list(cross_skills.keys()),
    "graph_stats": {
        "total_nodes": G.number_of_nodes(),
        "total_edges": G.number_of_edges()
    }
}

with open("../outputs/DRISHTI_FINAL_SUMMARY.json", 'w') as f:
    json.dump(final_summary, f, indent=2)

print("✅ Final summary saved!")
print(json.dumps(final_summary, indent=2))

✅ Final summary saved!
{
  "project": "DRISHTI",
  "total_videos": 7,
  "total_segments": 31,
  "unique_robot_skills": 4,
  "unique_intents": 3,
  "domains": [
    "shop",
    "artificial_jewellery"
  ],
  "cross_domain_transferable_skills": [],
  "graph_stats": {
    "total_nodes": 38,
    "total_edges": 31
  }
}
